In [1]:
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
import xgboost as xgb
from pycox.datasets import support

/home/ykzhou/anaconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


In [2]:
np.random.seed(1234)

In [3]:
df_train = support.read_df()
df_test = df_train.sample(frac=0.2)
df_train = df_train.drop(df_test.index)

In [4]:
cph = CoxPHFitter(penalizer=0.01, l1_ratio = 0)
cph.fit(df_train, duration_col='duration', event_col='event')
print('The C-statistics of Cox proportional hazards model is', 
      concordance_index(df_test['duration'], -cph.predict_partial_hazard(df_test), df_test['event']))

The C-statistics of Cox proportional hazards model is 0.5730882303445187


In [5]:
x = df_train.drop(columns = ['event', 'duration'])
y = df_train['duration'] * df_train['event'].replace({0: -1})
bst = xgb.XGBRegressor(objective='survival:cox').fit(x, y)
rs = bst.predict(x, output_margin=True)

df = df_train[['event', 'duration']]
df['XGBoost risk score'] = rs
cph = CoxPHFitter(penalizer = 0.01)
cph.fit(df, duration_col='duration', event_col='event')
    
x = df_test.drop(columns = ['event', 'duration'])
y = df_test['duration'] * df_test['event'].replace({0: -1})
rs = bst.predict(x, output_margin=True)
    
df = df_test[['event', 'duration']]
df['XGBoost risk score'] = rs
print('The C-statistics of Cox proportional hazards gradient boosting model is', 
      concordance_index(df['duration'], -cph.predict_partial_hazard(df), df['event']))

/home/ykzhou/anaconda3/lib/python3.8/site-packages/xgboost/data.py:262: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  elif isinstance(data.columns, (pd.Int64Index, pd.RangeIndex)):
/tmp/ipykernel_15676/477783774.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['XGBoost risk score'] = rs


The C-statistics of Cox proportional hazards gradient boosting model is 0.6144200925135983


/home/ykzhou/anaconda3/lib/python3.8/site-packages/xgboost/data.py:262: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  elif isinstance(data.columns, (pd.Int64Index, pd.RangeIndex)):
/tmp/ipykernel_15676/477783774.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['XGBoost risk score'] = rs
